# Entrega 2: Perfilado, Diccionario y Limpieza de Datos

Este notebook realiza la limpieza, formateo y transformación del dataset `Sumaria-2024.csv` de ENAHO, siguiendo las reglas establecidas en el diccionario de datos del proyecto.

## Unidad de análisis y granularidad

Unidad de análisis: hogar encuestado en ENAHO Sumaria 2024.

Granularidad: cada fila representa un hogar observado en un mes de levantamiento, con ubicación geográfica, condición socioeconómica, ingresos, gastos, composición del hogar y factor de expansión.

El dataset se usará como una tabla analítica a nivel de hogar para conectar posteriormente con Tableau.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. Carga de Datos
df_original = pd.read_csv('../Data/original/Sumaria-2024.csv', encoding='latin1')
print(f'Dimensiones originales: {df_original.shape}')

Dimensiones originales: (33691, 163)


## Perfilado inicial del dataset original

En esta sección se revisan dimensiones, tipos de variables, nulos, duplicados y cardinalidad inicial para identificar problemas de calidad antes de la limpieza.

In [3]:
print("Filas y columnas del dataset original:", df_original.shape)
print("Duplicados en dataset original:", df_original.duplicated().sum())

perfil_original = pd.DataFrame({
    "variable": df_original.columns,
    "tipo_original": df_original.dtypes.astype(str).values,
    "nulos": df_original.isnull().sum().values,
    "pct_nulos": (df_original.isnull().mean() * 100).round(2).values,
    "cardinalidad": df_original.nunique(dropna=True).values
})

perfil_original

Filas y columnas del dataset original: (33691, 163)
Duplicados en dataset original: 0


,variable,tipo_original,nulos,pct_nulos,cardinalidad
0,AÑO,int64,0,0.0,1
1,MES,int64,0,0.0,12
2,CONGLOME,int64,0,0.0,5359
3,VIVIENDA,int64,0,0.0,541
4,HOGAR,int64,0,0.0,15
...,...,...,...,...,...
158,FACTOR07,float64,0,0.0,968
159,LINEAV,float64,0,0.0,33488
160,POBREZAV,int64,0,0.0,4
161,NCONGLOME,int64,0,0.0,5323


## 2. Selección de Variables Relevantes
Nos quedamos estrictamente con las 18 variables definidas en la propuesta.

Estas variables fueron seleccionadas porque permiten responder la pregunta analítica del proyecto: analizar brechas socioeconómicas, diferencias territoriales, composición del gasto y variaciones temporales en hogares peruanos.

In [5]:
columnas_seleccionadas = [
    'MES', 'UBIGEO', 'DOMINIO', 'ESTRATO', 'POBREZA', 'POBREZAV',
    'INGHOG2D', 'GASHOG2D', 'MIEPERHO',
    'GRU11HD', 'GRU21HD', 'GRU31HD', 'GRU41HD', 'GRU51HD', 'GRU61HD', 'GRU71HD', 'GRU81HD',
    'FACTOR07'
]

df = df_original[columnas_seleccionadas].copy()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 33691 entries, 0 to 33690
Data columns (total 18 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   MES       33691 non-null  int64  
 1   UBIGEO    33691 non-null  int64  
 2   DOMINIO   33691 non-null  int64  
 3   ESTRATO   33691 non-null  int64  
 4   POBREZA   33691 non-null  int64  
 5   POBREZAV  33691 non-null  int64  
 6   INGHOG2D  33691 non-null  float64
 7   GASHOG2D  33691 non-null  float64
 8   MIEPERHO  33691 non-null  int64  
 9   GRU11HD   33691 non-null  float64
 10  GRU21HD   33691 non-null  float64
 11  GRU31HD   33691 non-null  float64
 12  GRU41HD   33691 non-null  float64
 13  GRU51HD   33691 non-null  float64
 14  GRU61HD   33691 non-null  float64
 15  GRU71HD   33691 non-null  float64
 16  GRU81HD   33691 non-null  float64
 17  FACTOR07  33691 non-null  float64
dtypes: float64(11), int64(7)
memory usage: 4.6 MB


## 2.1 Perfilado de variables seleccionadas

Se valida el subconjunto de variables que será usado en la limpieza inicial y en Tableau. Esta tabla permite revisar tipos, nulos, porcentaje de nulos y cardinalidad por campo.

In [6]:
print("Filas y columnas del dataset de trabajo:", df.shape)
print("Duplicados en variables seleccionadas:", df.duplicated().sum())

perfil_trabajo = pd.DataFrame({
    "variable": df.columns,
    "tipo": df.dtypes.astype(str).values,
    "nulos": df.isnull().sum().values,
    "pct_nulos": (df.isnull().mean() * 100).round(2).values,
    "cardinalidad": df.nunique(dropna=True).values
})

perfil_trabajo

Filas y columnas del dataset de trabajo: (33691, 18)
Duplicados en variables seleccionadas: 0


,variable,tipo,nulos,pct_nulos,cardinalidad
0,MES,int64,0,0.0,12
1,UBIGEO,int64,0,0.0,1271
2,DOMINIO,int64,0,0.0,8
3,ESTRATO,int64,0,0.0,8
4,POBREZA,int64,0,0.0,3
5,POBREZAV,int64,0,0.0,4
6,INGHOG2D,float64,0,0.0,33644
7,GASHOG2D,float64,0,0.0,33675
8,MIEPERHO,int64,0,0.0,17
9,GRU11HD,float64,0,0.0,33330


## 2.2 Campos críticos

| Campo | Rol analítico | Justificación |
|---|---|---|
| MES | Temporal | Permite analizar variaciones mensuales durante 2024. |
| UBIGEO | Geográfico | Permite ubicar territorialmente los hogares. |
| DOMINIO | Segmentación territorial | Permite comparar costa, sierra, selva y Lima Metropolitana. |
| ESTRATO | Segmentación poblacional | Permite comparar hogares según estrato. |
| POBREZA | Condición socioeconómica | Permite distinguir hogares pobres, pobres extremos y no pobres. |
| POBREZAV | Vulnerabilidad | Permite identificar hogares vulnerables o no vulnerables. |
| INGHOG2D | Métrica principal | Representa el ingreso del hogar. |
| GASHOG2D | Métrica principal | Representa el gasto del hogar. |
| MIEPERHO | Normalización | Permite considerar el tamaño del hogar. |
| GRU11HD-GRU81HD | Estructura del gasto | Permite analizar composición del consumo por rubro. |
| FACTOR07 | Ponderación | Permite aplicar el factor de expansión de la encuesta. |

## 3. Limpieza y Transformación (Mapping)

Reglas aplicadas:

1. **UBIGEO:** formato string de 6 dígitos con zero-padding. Se conserva como identificador geográfico, no como variable numérica.
2. **MES:** formato string de 2 dígitos con zero-padding para conservar consistencia temporal.
3. **Categóricas:** `DOMINIO`, `ESTRATO`, `POBREZA` y `POBREZAV` se homologan mediante diccionarios de etiquetas.
4. **Continuas:** ingresos, gastos, tamaño del hogar, grupos de gasto y factor de expansión se mantienen como variables numéricas.
5. **Duplicados y nulos:** se revisan antes y después de la limpieza.
6. **Outliers:** se aplica una revisión de valores extremos sobre `INGHOG2D` mediante IQR.
7. **Exportación:** se genera un CSV limpio preliminar compatible con Tableau.

In [7]:
# Formateo de Strings (UBIGEO y MES)
# Convertimos a string y si tienen .0 al final lo removemos, luego paddeamos con ceros a la izquierda
df['UBIGEO'] = df['UBIGEO'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(6)
df['MES'] = df['MES'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(2)

# Mapeo DOMINIO
dic_dominio = {
    1: 'Costa Norte', 2: 'Costa Centro', 3: 'Costa Sur',
    4: 'Sierra Norte', 5: 'Sierra Centro', 6: 'Sierra Sur',
    7: 'Selva', 8: 'Lima Metropolitana'
}
df['DOMINIO'] = df['DOMINIO'].map(dic_dominio)

# Mapeo ESTRATO
dic_estrato = {
    1: 'De 500,000 a más habitantes',
    2: 'De 100,000 a 499,999 habitantes',
    3: 'De 50,000 a 99,999 habitantes',
    4: 'De 20,000 a 49,999 habitantes',
    5: 'De 2,000 a 19,999 habitantes',
    6: 'De 500 a 1,999 habitantes',
    7: 'Área de Empadronamiento Rural (AER) Compuesto',
    8: 'Área de Empadronamiento Rural (AER) Simple'
}
df['ESTRATO'] = df['ESTRATO'].map(dic_estrato)

# Mapeo POBREZA
dic_pobreza = {
    1: 'Pobre Extremo',
    2: 'Pobre No Extremo',
    3: 'No Pobre'
}
df['POBREZA'] = df['POBREZA'].map(dic_pobreza)

# Mapeo POBREZAV
dic_pobrezav = {
    1: 'Pobre Extremo',
    2: 'Pobre No Extremo',
    3: 'Vulnerable No Pobre',
    4: 'No Vulnerable'
}
df['POBREZAV'] = df['POBREZAV'].map(dic_pobrezav)

# Asegurar numéricas continuas
num_cols = ['INGHOG2D', 'GASHOG2D', 'MIEPERHO', 'FACTOR07'] + [f'GRU{i}1HD' for i in range(1, 9)]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.head()

,MES,UBIGEO,DOMINIO,ESTRATO,POBREZA,POBREZAV,INGHOG2D,GASHOG2D,MIEPERHO,GRU11HD,GRU21HD,GRU31HD,GRU41HD,GRU51HD,GRU61HD,GRU71HD,GRU81HD,FACTOR07
0,01,010101,Sierra Norte,"De 20,000 a 49,999 habitantes",No Pobre,No Vulnerable,52162.609375,34188.218750,2,3845.363770,874.391602,1006.782715,1307.083496,3339.000000,3247.260742,1318.787598,3219.265869,79.816757
1,01,010101,Sierra Norte,"De 20,000 a 49,999 habitantes",No Pobre,No Vulnerable,40832.042969,40164.945312,3,12792.489258,2384.777832,916.000000,665.926880,2408.242676,2768.000000,895.084717,752.953064,79.816757
2,01,010101,Sierra Norte,"De 20,000 a 49,999 habitantes",No Pobre,No Vulnerable,15098.497070,12308.838867,1,2150.485107,603.583740,2589.000000,581.172546,2047.000000,233.000000,41.478962,557.595886,79.816757
3,01,010101,Sierra Norte,"De 20,000 a 49,999 habitantes",No Pobre,No Vulnerable,41082.953125,30316.724609,2,9690.755859,39.434135,3178.366211,2735.143311,652.000000,930.000000,0.000000,1418.503906,79.816757
4,01,010101,Sierra Norte,"De 20,000 a 49,999 habitantes",No Pobre,Vulnerable No Pobre,47659.160156,33076.910156,5,5630.693359,1683.596191,1686.000000,797.020874,1372.000000,3086.147705,103.000000,894.525024,79.816757


## 3.1 Validación posterior al mapping

Se revisa que el mapeo de categorías no haya generado valores nulos y que los tipos de datos sean adecuados para el análisis.

In [8]:
print("Nulos después del mapping:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nTipos después del mapping:")
print(df.dtypes)

print("\nCardinalidad después del mapping:")
print(df.nunique(dropna=True))

Nulos después del mapping:
Series([], dtype: int64)

Tipos después del mapping:
MES             str
UBIGEO          str
DOMINIO         str
ESTRATO         str
POBREZA         str
POBREZAV        str
INGHOG2D    float64
GASHOG2D    float64
MIEPERHO      int64
GRU11HD     float64
GRU21HD     float64
GRU31HD     float64
GRU41HD     float64
GRU51HD     float64
GRU61HD     float64
GRU71HD     float64
GRU81HD     float64
FACTOR07    float64
dtype: object

Cardinalidad después del mapping:
MES            12
UBIGEO       1271
DOMINIO         8
ESTRATO         8
POBREZA         3
POBREZAV        4
INGHOG2D    33644
GASHOG2D    33675
MIEPERHO       17
GRU11HD     33330
GRU21HD     23069
GRU31HD      8769
GRU41HD     31258
GRU51HD      6744
GRU61HD     22348
GRU71HD     16887
GRU81HD     31407
FACTOR07      968
dtype: int64


## 4. Tratamiento de Outliers

Se identifican valores atípicos en `INGHOG2D` que puedan distorsionar el análisis de promedios simples en visualizaciones preliminares de Tableau.

Se utilizará el método del Rango Intercuartílico (IQR).

Esta decisión debe interpretarse como una limpieza inicial. Dado que ENAHO puede contener hogares con ingresos altos reales, en futuras entregas se recomienda comparar resultados con y sin outliers, o evaluar una alternativa como winsorización.

In [10]:
def remover_outliers_iqr(dataset, columna):
    Q1 = dataset[columna].quantile(0.25)
    Q3 = dataset[columna].quantile(0.75)
    IQR = Q3 - Q1

    limite_inferior = max(0, Q1 - 1.5 * IQR)
    limite_superior = Q3 + 1.5 * IQR

    outliers = dataset[
        (dataset[columna] < limite_inferior) |
        (dataset[columna] > limite_superior)
    ]

    dataset_limpio = dataset[
        (dataset[columna] >= limite_inferior) &
        (dataset[columna] <= limite_superior)
    ].copy()

    porcentaje_removido = len(outliers) / len(dataset) * 100

    print(f"Variable evaluada: {columna}")
    print(f"Q1: {Q1:.2f}")
    print(f"Q3: {Q3:.2f}")
    print(f"IQR: {IQR:.2f}")
    print(f"Límite inferior: {limite_inferior:.2f}")
    print(f"Límite superior: {limite_superior:.2f}")
    print(f"Outliers removidos: {len(outliers)}")
    print(f"Porcentaje removido: {porcentaje_removido:.2f}%")

    return dataset_limpio


df_limpio = df.copy()

filas_antes = len(df_limpio)
print(f"Filas antes de limpieza: {filas_antes}")

df_limpio = remover_outliers_iqr(df_limpio, "INGHOG2D")

filas_despues = len(df_limpio)
print(f"Filas después de limpieza: {filas_despues}")
print(f"Filas eliminadas: {filas_antes - filas_despues}")

Filas antes de limpieza: 33691
Variable evaluada: INGHOG2D
Q1: 15495.03
Q3: 47507.70
IQR: 32012.67
Límite inferior: 0.00
Límite superior: 95526.70
Outliers removidos: 2037
Porcentaje removido: 6.05%
Filas después de limpieza: 31654
Filas eliminadas: 2037


## 5. Validación Final del Dataset Limpio

Se valida que el dataset limpio preliminar no tenga nulos, duplicados ni ambigüedades básicas de tipos antes de exportarlo para Tableau.

También se documenta la cardinalidad final de las variables para cumplir con el perfilado solicitado en la entrega.

In [11]:
print("Filas y columnas del dataset limpio antes de dropna:", df_limpio.shape)

nulos = df_limpio.isnull().sum()
print("Valores nulos por columna:")
print(nulos[nulos > 0])

duplicados_limpio = df_limpio.duplicated().sum()
print("\nDuplicados en dataset limpio:", duplicados_limpio)

# Si hubiera nulos por alguna conversión fallida, se eliminan para evitar problemas en Tableau.
df_limpio = df_limpio.dropna().copy()

print("\nFilas y columnas del dataset limpio después de dropna:", df_limpio.shape)
print("Duplicados finales:", df_limpio.duplicated().sum())

validacion_limpio = pd.DataFrame({
    "variable": df_limpio.columns,
    "tipo_limpio": df_limpio.dtypes.astype(str).values,
    "nulos": df_limpio.isnull().sum().values,
    "pct_nulos": (df_limpio.isnull().mean() * 100).round(2).values,
    "cardinalidad": df_limpio.nunique(dropna=True).values
})

validacion_limpio

Filas y columnas del dataset limpio antes de dropna: (31654, 18)
Valores nulos por columna:
Series([], dtype: int64)

Duplicados en dataset limpio: 0

Filas y columnas del dataset limpio después de dropna: (31654, 18)
Duplicados finales: 0


,variable,tipo_limpio,nulos,pct_nulos,cardinalidad
0,MES,str,0,0.0,12
1,UBIGEO,str,0,0.0,1271
2,DOMINIO,str,0,0.0,8
3,ESTRATO,str,0,0.0,8
4,POBREZA,str,0,0.0,3
5,POBREZAV,str,0,0.0,4
6,INGHOG2D,float64,0,0.0,31607
7,GASHOG2D,float64,0,0.0,31638
8,MIEPERHO,int64,0,0.0,17
9,GRU11HD,float64,0,0.0,31303


## 5.1 Modelo Analítico Inicial para Tableau

Para esta entrega se utiliza una tabla plana a nivel de hogar. No se aplican joins ni relaciones adicionales porque el análisis inicial se concentra en una sola fuente: ENAHO Sumaria 2024.

La tabla limpia funciona como base analítica preliminar para Tableau.

Dimensiones principales:
- `MES`
- `UBIGEO`
- `DOMINIO`
- `ESTRATO`
- `POBREZA`
- `POBREZAV`

Métricas principales:
- `INGHOG2D`
- `GASHOG2D`
- `MIEPERHO`
- `GRU11HD` a `GRU81HD`
- `FACTOR07`

Validación de cardinalidad esperada:
- `MES` debe tener 12 valores.
- `DOMINIO` debe tener 8 categorías.
- `ESTRATO` debe tener 8 categorías.
- `POBREZA` debe tener 3 categorías.
- `POBREZAV` debe tener 4 categorías.
- `UBIGEO` debe mantenerse como texto para conservar códigos territoriales.

## 6. Exportación del Dataset Limpio

In [ ]:
import os
import csv
os.makedirs('../Data/limpio', exist_ok=True)

# Forzar el zfill justo antes de exportar para evitar cualquier pérdida del tipo string
df_limpio['UBIGEO'] = df_limpio['UBIGEO'].astype(str).str.zfill(6)
df_limpio['MES'] = df_limpio['MES'].astype(str).str.zfill(2)

# Exportamos con codificación utf-8-sig (con BOM) para asegurar compatibilidad de tildes en Excel/Tableau
# Usamos QUOTE_NONNUMERIC para forzar comillas en columnas de tipo string (como UBIGEO)
df_limpio.to_csv('../Data/limpio/Sumaria-2024_limpio.csv', index=False, encoding='utf-8-sig', quoting=csv.QUOTE_NONNUMERIC)
print('Dataset exportado exitosamente a Data/limpio/Sumaria-2024_limpio.csv')

## 7. Conclusión de Entrega 2

El dataset limpio preliminar queda preparado para su conexión inicial en Tableau. La entrega cubre perfilado, selección de variables, revisión de nulos, duplicados, cardinalidad, homologación de categorías, tratamiento inicial de outliers, bitácora de transformaciones y exportación del archivo limpio.

Las principales decisiones metodológicas fueron:

- trabajar a nivel de hogar;
- conservar variables temporales, geográficas, socioeconómicas y de gasto;
- preservar `MES` y `UBIGEO` como texto;
- usar una tabla plana inicial para Tableau;
- aplicar una limpieza preliminar de outliers sobre ingresos del hogar.

